In [1]:
import polars as pl
import psycopg2
import dagster as dg

In [ ]:
class PostgresResource():
    """Configuration schema for PostgreSQL resource."""
    def __init__(self, host: str, port: int, database: str, user: str, password: str, chunk_size: int):
        self.host: str = host
        self.port: int = port
        self.database: str = database
        self.user: str = user
        self.password: str = password
        self.chunk_size: int = chunk_size | 1_000_000
        
    def _get_connection(self) -> psycopg2.extensions.connection:
            """Creates a psycopg2 connection to PostgreSQL."""
            try:
                conn = psycopg2.connect(
                    dbname=self.database,
                    user=self.user,
                    host=self.host,
                    password=self.password,
                    port=self.port
                )
                return conn
            except Exception as e:
                print(f"Unable to connect to the database: {str(e)}")
                raise
    
    def _get_connection_string(self) -> str:
        """Constructs a PostgreSQL connection string."""
        return f"postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}"

    def empty_table(self, table_name: str, schema: str) -> None:
        """Truncate the specified table."""
        conn = self._get_connection()
                
        try:
            with conn.cursor() as cur:
                full_table_name = f"{schema}.{table_name}"
                cur.execute(f"TRUNCATE TABLE {full_table_name} CASCADE")
                conn.commit()
        except Exception as e:
            print(f"Failed to truncate table: {str(e)}")
            raise
        finally:
            conn.close()
    
    def get_table_columns(self, table_name: str, schema: str) -> list:
        """Get column names of the specified table."""
        conn = self._get_connection()
        try:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT column_name
                    FROM information_schema.columns
                    where 
                        table_schema='{schema}'
                        and
                        table_name='{table_name}'
                    order by ordinal_position;
                    """
                    
                )
                columns = [row[0] for row in cur.fetchall()]
                return columns
        except Exception as e:
            print(f"Failed to get table columns: {str(e)}")
            raise
        finally:
            conn.close()

    def _check_table_columns(self, table_name: str, schema: str, df: pl.DataFrame) -> None:
        """Check if DataFrame columns match the target table columns."""
        table_columns = self.get_table_columns(table_name, schema)
        df_columns = df.columns
        
        missing_columns = set(table_columns) - set(df_columns)
        extra_columns = set(df_columns) - set(table_columns)
        
        if missing_columns:
            raise ValueError(f"Missing columns in DataFrame: {missing_columns}")
        if extra_columns:
            raise ValueError(f"Extra columns in DataFrame: {extra_columns}")
    
    def load_polars_dataframe(self, df: pl.DataFrame, table_name: str, schema: str) -> None:
        """Load a Polars DataFrame to PostgreSQL in chunks."""
        try:
            self.empty_table(table_name, schema)
            total_chunks = (df.height + self.chunk_size - 1) // self.chunk_size

            connection_string = self._get_connection_string()

            for offset in range(0, df.height, self.chunk_size):
                message = f"Loading chunk {offset // self.chunk_size + 1} of {total_chunks}..."
                print(message)
                batch = df.slice(offset, self.chunk_size)
                batch.write_database(
                    table_name=f"{schema}.{table_name}",
                    if_table_exists="append",
                    connection=connection_string
                )
        except Exception as e:
            print(f"Failed to load data: {str(e)}")
            raise

    def get_query_results(self, query: str) -> pl.DataFrame:
        """Execute a SQL query and return results as a Polars DataFrame."""
        conn = self._get_connection()
        try:
            with conn.cursor() as cur:
                cur.execute(query)
                columns = [desc[0] for desc in cur.description]
                data = cur.fetchall()
                return pl.DataFrame(data, schema=columns)
        except Exception as e:
            print(f"Failed to execute query: {str(e)}")
            raise
        finally:
            conn.close()


In [ ]:

POSTGRES_USER=""
POSTGRES_PASSWORD=""
DATABASE=""
HOST=""
PORT=""
IMDB_SCHEMA=""

test = PostgresResource(
    host=HOST,
    port=PORT,
    database=DATABASE,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    chunk_size=1_000_000
)

In [4]:
path = "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.ratings.tsv"
df = pl.read_csv(path, separator="\t")
df = pl.read_csv(
        path,
        has_header=True,
        separator="\t",
        truncate_ragged_lines=True,
        null_values="\\N",
        quote_char=None,
        schema={
            "tconst": pl.Utf8,
            "averageRating": pl.Float16,
            "numVotes": pl.UInt32,
        },
    )
df = df.rename({"averageRating": "average_rating", "numVotes":"num_votes"})
# test.empty_table("title_ratings", IMDB_SCHEMA)
test.load_polars_dataframe(df, "title_ratings", IMDB_SCHEMA)

Loading chunk 1 of 2...
Loading chunk 2 of 2...


In [5]:
results = test.get_query_results(f"SELECT * FROM {IMDB_SCHEMA}.title_ratings LIMIT 5")
results

/tmp/ipykernel_162603/1520272584.py:112: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return pl.DataFrame(data, schema=columns)


tconst,average_rating,num_votes
str,"decimal[38,1]",i64
"""tt0000001""",5.7,2200
"""tt0000002""",5.5,311
"""tt0000003""",6.4,2308
"""tt0000004""",5.1,196
"""tt0000005""",6.2,3037
